# 2. PyTorch 张量与自动微分

本节目标：理解 Tensor 的创建与广播，掌握梯度的计算、读取、清零和计算图分离；学会用 `help()`、`dir()` 查资料。

沿用前一本笔记的层级：二级标题划分主题，三级标题讲具体知识点，代码紧跟解释。
**建议从上往下运行；所有例子使用 CPU 即可，不依赖上一本 Notebook 的变量。**

In [1]:
import numpy as np
import torch

print("PyTorch：", torch.__version__)
print("默认浮点类型：", torch.get_default_dtype())

PyTorch： 2.15.0
默认浮点类型： torch.float32


## 2.1 Tensor：创建、属性与广播
### 2.1.1 什么是张量？怎样创建？

Tensor（张量）可以先理解为 PyTorch 中组织数值的多维数组；它支持自动微分，也可以放到支持的加速设备上计算。

- `np.array(...)` 创建 **NumPy 数组**。
- `torch.tensor(...)` 创建 **PyTorch 张量**。
- `dtype` 指元素类型；`shape` 指各维度的长度；`device` 指数据所在设备。

| 写法 | 含义 | 形状 |
|---|---|---|
| `torch.tensor(3.0)` | 标量，零维 | `()` |
| `torch.tensor([1.0, 2.0, 3.0])` | 一维数组 | `(3,)` |
| `torch.tensor([[1.0, 2.0, 3.0]])` | 二维，1 行 3 列 | `(1, 3)` |

`torch.Size([1, 3])` 是 PyTorch 的形状显示方式，可用 `tuple(x.shape)` 转为元组查看。

In [2]:
scalar = torch.tensor(3.0)
vector = torch.tensor([1.0, 2.0, 3.0])
matrix = torch.tensor([[1.0, 2.0, 3.0]], dtype=torch.float32)

for name, value in [("scalar", scalar), ("vector", vector), ("matrix", matrix)]:
    print(name, "shape=", tuple(value.shape), "ndim=", value.ndim,
          "numel=", value.numel(), "dtype=", value.dtype, "device=", value.device)

scalar shape= () ndim= 0 numel= 1 dtype= torch.float32 device= cpu
vector shape= (3,) ndim= 1 numel= 3 dtype= torch.float32 device= cpu
matrix shape= (1, 3) ndim= 2 numel= 3 dtype= torch.float32 device= cpu


### 2.1.2 默认类型与常用创建方法

在没有修改全局默认设置的常见环境中：

- `torch.tensor([1, 2])` 根据整数数据推断为 `torch.int64`。
- `torch.tensor([1.0, 2.0])` 使用默认浮点类型，通常是 `torch.float32`。
- `np.array([1.0, 2.0])` 通常是 `float64`；`rng.uniform(...)` 也是 `float64`。
- `torch.from_numpy(...)` **保留 NumPy 数组类型**，不会自动转成 `float32`。

学习项目中，连续数值输入和模型参数先统一用 `float32`；分类类别编号通常用 `int64`。
常见创建方法还有 `zeros`、`ones`、`arange`、`randn`、`zeros_like`。

In [3]:
print("整数推断：", torch.tensor([1, 2]).dtype)
print("浮点推断：", torch.tensor([1.0, 2.0]).dtype)
print("全零：\n", torch.zeros((2, 3), dtype=torch.float32))
print("全一：\n", torch.ones((2, 3)))
print("等间隔整数：", torch.arange(0, 6))
print("同形状全零：", torch.zeros_like(matrix))

generator = torch.Generator().manual_seed(42)
print("标准正态随机数：\n", torch.randn((2, 3), generator=generator))

整数推断： torch.int64
浮点推断： torch.float32
全零：
 tensor([[0., 0., 0.],
        [0., 0., 0.]])
全一：
 tensor([[1., 1., 1.],
        [1., 1., 1.]])
等间隔整数： tensor([0, 1, 2, 3, 4, 5])
同形状全零： tensor([[0., 0., 0.]])
标准正态随机数：
 tensor([[ 0.3367,  0.1288,  0.2345],
        [ 0.2303, -1.1229, -0.1863]])


### 2.1.3 从 NumPy 转换：类型与共享内存

```python
torch.tensor(array)       # 复制数据，通常保留数组 dtype，可显式指定 dtype
torch.from_numpy(array)   # 与支持的 NumPy 数组共享内存，并保留 dtype
x.to(dtype=torch.float32) # 转换类型，使用返回结果
```

共享内存意味着改一边的数据，另一边也可能变化。先知道这个区别，不需要在初学时追求省一次复制。
`torch.tensor(existing_tensor)` 会复制并切断原计算历史；已有张量的复制与分离后面用 `clone()`、`detach()` 明确表达。

In [4]:
array_np = np.array([1.0, 2.0], dtype=np.float64)
shared = torch.from_numpy(array_np)
copied = torch.tensor(array_np, dtype=torch.float32)
array_np[0] = 99.0

print("NumPy：", array_np)
print("共享：", shared, shared.dtype)
print("复制：", copied, copied.dtype)
assert shared[0].item() == 99.0
assert copied[0].item() == 1.0

NumPy： [99.  2.]
共享： tensor([99.,  2.], dtype=torch.float64) torch.float64
复制： tensor([1., 2.]) torch.float32


### 2.1.4 形状与常见运算

- `x.shape`：形状；`x.ndim`：维度个数；`x.numel()`：元素总数。
- `reshape(...)` 改变组织形状，元素总数必须一致；返回结果可能与原张量共享数据。
- `*` 是逐元素乘法；`@` 是矩阵乘法（这里先讨论二维输入）。
- 整数索引移除对应维度，切片保留对应维度；规则与此前学的 NumPy 一致。

In [5]:
grid = torch.arange(1, 7, dtype=torch.float32).reshape(2, 3)
print("grid：\n", grid)
print("取一行：", grid[0], grid[0].shape)
print("保留行维度：", grid[0:1, :], grid[0:1, :].shape)
print("逐元素平方：\n", grid * grid)
print("矩阵乘法形状：", (grid @ grid.T).shape)  # (2,3) @ (3,2) -> (2,2)

grid：
 tensor([[1., 2., 3.],
        [4., 5., 6.]])
取一行： tensor([1., 2., 3.]) torch.Size([3])
保留行维度： tensor([[1., 2., 3.]]) torch.Size([1, 3])
逐元素平方：
 tensor([[ 1.,  4.,  9.],
        [16., 25., 36.]])
矩阵乘法形状： torch.Size([2, 2])


### 2.1.5 广播机制：从右往左比较形状

广播允许某些不同形状的张量进行逐元素运算。它按规则对齐维度，**不是任意形状都能相加**。

1. 从最右边的维度开始比较。
2. 对应维度长度相同，或者其中一个是 `1`，才兼容。
3. 缺失的前导维度视为 `1`。
4. 结果取每个兼容维度展开后的长度；初学时可理解为把长度为 `1` 的维度虚拟扩展。

例如：

```text
(2, 3)
   (3,)  -> 按 (1, 3) 对齐
--------
(2, 3)
```

`(2, 3)` 和 `(2,)` 不兼容，因为最右侧的 `3` 和 `2` 既不相等，也没有一个为 `1`。

In [6]:
base = torch.tensor([[1.0, 2.0, 3.0],
                     [4.0, 5.0, 6.0]])
row_bias = torch.tensor([10.0, 20.0, 30.0])
column_bias = torch.tensor([[100.0], [200.0]])

print("加标量：\n", base + 1)
print("每行加同一个向量：\n", base + row_bias)
print("每行分别加一个数：\n", base + column_bias)
assert (base + row_bias).shape == (2, 3)
assert (base + column_bias).shape == (2, 3)

try:
    base + torch.tensor([10.0, 20.0])
except RuntimeError:
    print("预期错误：(2,3) 与 (2,) 不能直接广播相加。")

加标量：
 tensor([[2., 3., 4.],
        [5., 6., 7.]])
每行加同一个向量：
 tensor([[11., 22., 33.],
        [14., 25., 36.]])
每行分别加一个数：
 tensor([[101., 102., 103.],
        [204., 205., 206.]])
预期错误：(2,3) 与 (2,) 不能直接广播相加。


### 2.1.6 训练中的广播陷阱：能运行不代表算对

预测为 `(N,1)`，标签为 `(N,)` 时，相减结果可能是 `(N,N)`，变成了所有预测和标签的两两比较！

```text
(N, 1)
   (N,)  -> (1, N)
--------
(N, N)
```

计算逐样本 MSE 前，应检查预测与标签形状一致。这里展示的是列向量回归任务；其他任务的损失函数可能有不同的形状约定。

In [7]:
prediction = torch.tensor([[1.0], [2.0], [3.0]])
target_wrong_shape = torch.tensor([1.0, 2.0, 3.0])

wrong_error = prediction - target_wrong_shape
print("错误广播的差值：\n", wrong_error)
print("错误形状：", wrong_error.shape)
print("错误的 MSE：", wrong_error.square().mean().item())

target = target_wrong_shape.reshape(-1, 1)  # -1 表示自动推断这一维
assert prediction.shape == target.shape
correct_error = prediction - target
print("正确形状：", correct_error.shape)
print("正确的 MSE：", correct_error.square().mean().item())
assert wrong_error.shape == (3, 3)
assert correct_error.square().mean().item() == 0.0

错误广播的差值：
 tensor([[ 0., -1., -2.],
        [ 1.,  0., -1.],
        [ 2.,  1.,  0.]])
错误形状： torch.Size([3, 3])
错误的 MSE： 1.3333333730697632
正确形状： torch.Size([3, 1])
正确的 MSE： 0.0


## 2.2 自动微分与计算图
### 2.2.1 requires_grad、计算图和链式法则

自动微分不是凭空猜梯度，而是跟踪运算，并按链式法则计算导数。
对初学例子，可以理解为：PyTorch 记录前向计算关系，再沿关系反向计算梯度。

$$
x \longrightarrow u=x^2 \longrightarrow L=3u
$$

在 $x=2$ 时：

$$
\frac{dL}{dx}=\frac{dL}{du}\frac{du}{dx}=3\times2x=12
$$

- `requires_grad=True`：要求跟踪这个张量相关运算的梯度。
- `grad_fn`：非叶子结果对应的反向计算信息；手动创建的叶子张量通常为 `None`。
- `is_leaf`：是否是叶子张量；可训练参数通常是叶子张量。
- 需要梯度的张量必须使用浮点或复数类型；整数张量不能直接设置 `requires_grad=True`。

In [8]:
x = torch.tensor(2.0, requires_grad=True)
u = x ** 2
loss = 3 * u

print("x：", x, "is_leaf：", x.is_leaf, "grad_fn：", x.grad_fn)
print("u：", u, "is_leaf：", u.is_leaf, "grad_fn：", u.grad_fn)
print("loss：", loss)

loss.backward()
print("x 的梯度：", x.grad)  # 12
assert torch.allclose(x.grad, torch.tensor(12.0))

x： tensor(2., requires_grad=True) is_leaf： True grad_fn： None
u： tensor(4., grad_fn=<PowBackward0>) is_leaf： False grad_fn： <PowBackward0 object at 0x105dab6d0>
loss： tensor(12., grad_fn=<MulBackward0>)
x 的梯度： tensor(12.)


### 2.2.2 backward() 与 .grad 分别做什么？

正确拼写是 **`backward()`**。

- `loss.backward()`：从损失出发执行反向传播，计算梯度。
- `x.grad`：读取累计到张量上的梯度。它是属性，不写成 `.grad()`。
- `backward()` **不更新参数**，更新是手动运算或 `optimizer.step()` 的工作。
- 默认会为参与计算且 `requires_grad=True` 的叶子张量保留 `.grad`。
- 非叶子中间结果默认不保留 `.grad`；确实需要查看时，在反向传播前调用 `retain_grad()`。
- 反向传播前 `.grad` 通常为 `None`，它不同于梯度数值恰好为零。

In [9]:
leaf = torch.tensor(2.0, requires_grad=True)
intermediate = leaf ** 2
intermediate.retain_grad()  # 为教学观察显式保留中间梯度
objective = 3 * intermediate

print("反向前 leaf.grad：", leaf.grad)
objective.backward()
print("leaf.grad：", leaf.grad)                  # dL/dx = 12
print("intermediate.grad：", intermediate.grad)  # dL/du = 3
print("leaf 的值没被更新：", leaf.item())        # 仍然为 2
assert leaf.item() == 2.0
assert intermediate.grad.item() == 3.0

反向前 leaf.grad： None
leaf.grad： tensor(12.)
intermediate.grad： tensor(3.)
leaf 的值没被更新： 2.0


### 2.2.3 向量输出如何反向传播？

对只含一个元素的损失，可直接 `loss.backward()`，相当于从导数种子 `1` 开始。
对多个元素的输出，PyTorch 不知道你想怎样汇总，需要先聚合成标量，或显式传入同形状的权重。

```python
y.sum().backward()                # 求所有分量之和的梯度
y.backward(torch.ones_like(y))    # 等价的全 1 权重方式（应使用新的计算图）
```

一般权重得到的是加权和的梯度，也叫向量—雅可比积；它不是直接返回完整雅可比矩阵。当前先掌握标量损失的用法。

In [10]:
vector_x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
vector_y = vector_x ** 2
vector_y.sum().backward()
print("sum 的梯度：", vector_x.grad)  # [2,4,6]

weighted_x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
weights = torch.tensor([1.0, 0.0, 2.0])
(weighted_x ** 2).backward(weights)
print("加权后的梯度：", weighted_x.grad)  # [2,0,12]
assert torch.allclose(weighted_x.grad, torch.tensor([2.0, 0.0, 12.0]))

sum 的梯度： tensor([2., 4., 6.])
加权后的梯度： tensor([ 2.,  0., 12.])


### 2.2.4 梯度为什么需要清零？

`backward()` 会把新梯度**累加**到已有 `.grad` 上，而非每次自动覆盖。
这支持跨多个小批次进行梯度累积，但普通训练通常每轮需要先清掉旧梯度。

- `x.grad.zero_()`：将已有梯度原地置零，要求 `.grad` 不是 `None`。
- `x.grad = None`：让下一次反向传播重新建立梯度。
- `optimizer.zero_grad(set_to_none=True)`：训练时常用的清理方式；未参与运算的参数可能保持 `None`。

本例每次都重新前向计算，从而创建新计算图；旧梯度仍会保留，直到显式清理。

In [11]:
acc_x = torch.tensor(3.0, requires_grad=True)
(acc_x ** 2).backward()
print("第一次：", acc_x.grad)  # 6

(acc_x ** 2).backward()  # 新前向、新图，但同一个叶子张量
print("第二次累积：", acc_x.grad)  # 12
assert acc_x.grad.item() == 12.0

acc_x.grad.zero_()
(acc_x ** 2).backward()
print("清零再求导：", acc_x.grad)  # 6
assert acc_x.grad.item() == 6.0

第一次： tensor(6.)
第二次累积： tensor(12.)
清零再求导： tensor(6.)


### 2.2.5 梯度累积与“重复同一计算图”不是一回事

一次 `backward()` 默认会释放反向计算需要的已保存中间量。同一个计算结果再次调用 `backward()`，常会报“第二次反向传播”的错误。

普通训练每轮重新进行前向计算，建立新图即可。
`retain_graph=True` 会保留用于反向的图数据，适合确有需要的多次反向计算，但会占用更多内存；**不要把它当作普通训练的通用补丁**。
`create_graph=True` 则用于构建梯度本身的计算图、计算高阶导数，现阶段了解名字即可。

In [12]:
graph_x = torch.tensor(3.0, requires_grad=True)
graph_loss = graph_x ** 2
graph_loss.backward()
try:
    graph_loss.backward()
except RuntimeError:
    print("预期报错：这个计算图需要的中间量已被释放。")

# 正常做法：清梯度，重新前向，再反向。
graph_x.grad = None
new_graph_loss = graph_x ** 2
new_graph_loss.backward()
print("重新前向后的梯度：", graph_x.grad)

预期报错：这个计算图需要的中间量已被释放。
重新前向后的梯度： tensor(6.)


### 2.2.6 分离计算：detach()、no_grad()、item()

这里把“分离计算”理解为切断自动微分关系或暂停跟踪；它不是文件模块的拆分。

| 写法 | 作用 | 常见用途 |
|---|---|---|
| `y.detach()` | 返回与当前计算图分离的新张量；不再沿此分支回传 | 固定某分支、记录数值、转换到 NumPy |
| `with torch.no_grad():` | 区域内通常不记录新运算的梯度关系 | 参数更新、验证/推理 |
| `loss.item()` | 单元素张量转 Python 数值，不保留可微关系 | 打印、保存损失曲线 |
| `y.detach().cpu().numpy()` | 分离后转 CPU，再转 NumPy | 可视化、数值分析 |

**注意：**
- `detach()` 不会修改原张量，也不复制数据；分离结果与原张量共享存储。
- 同时需要切断梯度并独立复制数据时，用 `y.detach().clone()`。
- `clone()` 本身一般保留梯度关系，不等同于 `detach()`。
- `no_grad()` 不会把区域外已有参数的 `requires_grad` 改成 `False`。
- `model.eval()` 改变 Dropout/BatchNorm 等层的行为，**不等于**关闭梯度。
- 不用 `.data` 绕过梯度跟踪来更新参数，推荐 `no_grad()`。

In [13]:
detach_x = torch.tensor(2.0, requires_grad=True)
branch = detach_x ** 2
detached_branch = branch.detach()

# 分离分支按常数看待；只对另一条 3*x 的分支求导。
detached_loss = detached_branch + 3 * detach_x
detached_loss.backward()
print("含 detach 分支的梯度：", detach_x.grad)  # 3，不是 7
assert detach_x.grad.item() == 3.0

with torch.no_grad():
    untracked = detach_x * 10
print("no_grad 结果是否需梯度：", untracked.requires_grad)
print("原参数是否需梯度：", detach_x.requires_grad)
print("NumPy 数值：", branch.detach().cpu().numpy())

含 detach 分支的梯度： tensor(3.)
no_grad 结果是否需梯度： False
原参数是否需梯度： True
NumPy 数值： 4.0


In [14]:
storage_source = torch.tensor([1.0, 2.0], requires_grad=True)
shared_detached = storage_source.detach()
independent = storage_source.detach().clone()

# 只演示共享存储；这个例子后面不做反向传播。
shared_detached[0] = 99.0
print("原张量也变了：", storage_source)
print("独立副本未改变：", independent)
assert storage_source[0].item() == 99.0
assert independent[0].item() == 1.0

# 实际训练中不要随意修改 detach 得到的共享数据，可能破坏反向传播所需数值。

原张量也变了： tensor([99.,  2.], requires_grad=True)
独立副本未改变： tensor([1., 2.])


### 2.2.7 连接最优化：手算梯度与自动微分核对

线性回归模型与均方误差：

$$
\hat y_i=wx_i+b,\qquad L=\frac{1}{N}\sum_{i=1}^{N}(wx_i+b-y_i)^2
$$

$$
\frac{\partial L}{\partial w}=\frac{2}{N}\sum_{i=1}^{N}x_i(wx_i+b-y_i),\qquad
\frac{\partial L}{\partial b}=\frac{2}{N}\sum_{i=1}^{N}(wx_i+b-y_i)
$$

更新规则：$w\leftarrow w-\eta\,\partial L/\partial w$，$b\leftarrow b-\eta\,\partial L/\partial b$。
下面先计算两个旧参数处的梯度，再同时使用它们更新；不在更新第一个参数后重新计算第二个梯度。

In [15]:
train_x = torch.tensor([[1.0], [2.0], [3.0]])
train_y = 3 * train_x + 2
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

pred = w * train_x + b
assert pred.shape == train_y.shape
train_loss = (pred - train_y).square().mean()
train_loss.backward()

with torch.no_grad():
    error = pred - train_y
    manual_w = 2 * (train_x * error).mean()
    manual_b = 2 * error.mean()
print("自动梯度 w、b：", w.grad.item(), b.grad.item())
print("手算梯度 w、b：", manual_w.item(), manual_b.item())
assert torch.allclose(w.grad, manual_w)
assert torch.allclose(b.grad, manual_b)

before = train_loss.item()
with torch.no_grad():
    w -= 0.05 * w.grad
    b -= 0.05 * b.grad
    after = ((w * train_x + b - train_y) ** 2).mean().item()
w.grad = None
b.grad = None
print("更新后参数：", w.item(), b.item())
print("损失变化：", before, "->", after)
assert after < before

自动梯度 w、b： -36.0 -16.0
手算梯度 w、b： -36.0 -16.0
更新后参数： 1.8000000715255737 0.800000011920929
损失变化： 70.0 -> 13.919998168945312


## 2.3 查资料的两个工具：dir() 与 help()
### 2.3.1 dir()：这个对象有哪些名字？

`dir(object)` 返回对象提供的属性/方法名称列表，适合发现和查找接口；它不告诉你每个接口怎样使用。

```python
dir(torch)   # torch 模块有哪些名字？
dir(x)       # 张量对象有哪些名字？
```

名称两边有双下划线的通常是 Python 特殊接口，初学阶段优先关注普通名称。

In [16]:
lookup_tensor = torch.tensor([[1.0, 2.0]])
public_names = [name for name in dir(lookup_tensor) if not name.startswith("_")]
print("部分公开名称：", public_names[:20])
print("名字含 grad：", [name for name in public_names if "grad" in name])
print("是否有 reshape：", "reshape" in dir(lookup_tensor))

部分公开名称： ['H', 'T', 'abs', 'abs_', 'absolute', 'absolute_', 'acos', 'acos_', 'acosh', 'acosh_', 'add', 'add_', 'addbmm', 'addbmm_', 'addcdiv', 'addcdiv_', 'addcmul', 'addcmul_', 'addmm', 'addmm_']
名字含 grad： ['grad', 'grad_dtype', 'grad_fn', 'register_post_accumulate_grad_hook', 'requires_grad', 'requires_grad_', 'retain_grad', 'retains_grad']
是否有 reshape： True


### 2.3.2 help()：怎样使用这个接口？

`help(...)` 显示对象的说明，包括可用时的参数、返回值与示例。内容较长时只需先找当前需要的部分。

- `help(torch.tensor)`：查创建张量。
- `help(torch.Tensor.backward)`：查反向传播。
- `help(lookup_tensor.reshape)`：查张量方法。
- `help(torch.numel)`：短文档例子，查询元素总数。

传入的是**函数/方法本身**，先不要执行它：

```python
help(torch.tensor)       # 查函数说明
# help(torch.tensor(...)) 先调用创建函数，再查看结果对象的说明，不是同一件事
```

Notebook 也可用 `torch.tensor?` 查看文档；该问号语法是 IPython 功能，不是普通 `.py` 的 Python 语法。

In [17]:
help(torch.numel)

Help on built-in function numel in module torch:

numel(...)
    numel(input: Tensor) -> int
    
    Returns the total number of elements in the :attr:`input` tensor.
    
    Args:
        input (Tensor): the input tensor.
    
    Example::
    
        >>> a = torch.randn(1, 2, 3, 4, 5)
        >>> torch.numel(a)
        120
        >>> a = torch.zeros(4,4)
        >>> torch.numel(a)
        16



### 2.3.3 属性与方法：什么时候加括号？

```python
lookup_tensor.shape       # 属性：直接读取
lookup_tensor.dtype       # 属性
lookup_tensor.grad        # 属性，尚未生成时可能为 None
lookup_tensor.numel()     # 方法：调用并获取元素总数
lookup_tensor.reshape(2, 1)  # 方法：带参数调用
```

查看顺序：先明确目标 → 用 `dir()` 找候选名称 → 用 `help()` 查用法 → 写最小例子验证。
API 的参数与边界行为还可以对照 [PyTorch 官方文档](https://docs.pytorch.org/docs/stable/index.html)。

In [18]:
print("形状：", lookup_tensor.shape)
print("类型：", lookup_tensor.dtype)
print("元素数：", lookup_tensor.numel())
print("reshape 文档开头：\n", lookup_tensor.reshape.__doc__.strip().split("\n\n")[0])

形状： torch.Size([1, 2])
类型： torch.float32
元素数： 2
reshape 文档开头：
 reshape(*shape) -> Tensor


## 2.4 复习与小练习

先预测，必要时再用新单元格验证：

1. `(4,1)` 和 `(3,)` 相加，结果形状是什么？为什么？
2. 预测 `(N,1)`、标签 `(N,)` 能相减，为什么仍可能写错损失？
3. $x=3$，$L=2x^2+x$，`L.backward()` 后 `x.grad` 应是多少？
4. 不清梯度、但每次重新前向，再求一次同样的梯度，`.grad` 会发生什么？
5. `u=x**2`，`L=u.detach()+3*x`，梯度沿哪些分支传播？
6. `help(torch.tensor)` 和 `dir(torch)` 各自回答哪类问题？

**掌握标准：**能解释 shape/dtype、判断简单广播、用自动微分核对导数、正确清理梯度，并区分分离与复制。

<details>
<summary>完成后再展开答案</summary>

1. `(4,3)`，右侧对齐后 `(4,1)` 与 `(1,3)` 兼容。
2. 会广播为 `(N,N)` 的两两差值，不是预期逐样本的 `(N,1)` 差值。
3. $4x+1=13$。
4. 在已有梯度上累加；同样的计算做两次，通常为一次梯度的两倍。
5. `u.detach()` 这一分支被当作常数，只从 `3*x` 贡献梯度，结果为 3。
6. `help` 查具体对象用法，`dir` 列出可用名称。

</details>

本节不要求掌握高阶微分或自定义反向算子，先把训练循环里的基础用法练熟。